# 🧠 Cerebro LLM Training on Google Colab

**GPU-accelerated training pipeline for the Cerebro language model.**

This notebook trains Cerebro on HuggingFace datasets using Colab's free GPU (T4/V100).

## Setup Instructions

1. Upload the Cerebro project to your Google Drive (see cell below)
2. Run each cell in order
3. Training starts automatically on GPU

## Step 0: Get Cerebro Code onto Google Drive

**Option A (Recommended):** Upload the entire `cerebro/` project folder to your Google Drive root.
- Zip the `cerebro/` directory on your local machine
- Upload `cerebro.zip` to Google Drive
- Unzip in Drive (right-click → Open with → Zip Extractor)

**Option B:** Clone from GitHub (once repo is public):
```python
!git clone https://github.com/YOUR_USERNAME/cerebro-llm.git cerebro
```

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy Cerebro from Google Drive to Colab runtime
import os
import shutil

CEREBRO_SOURCE = '/content/drive/MyDrive/cerebro'  # Adjust if needed
CEREBRO_DEST = '/content/cerebro'

if os.path.exists(CEREBRO_SOURCE):
    if os.path.exists(CEREBRO_DEST):
        shutil.rmtree(CEREBRO_DEST)
    shutil.copytree(CEREBRO_SOURCE, CEREBRO_DEST)
    print(f'Copied Cerebro from Drive to {CEREBRO_DEST}')
    print(f'Files: {len(os.listdir(CEREBRO_DEST))}')
else:
    print(f'ERROR: Cerebro not found at {CEREBRO_SOURCE}')
    print('Upload the cerebro project folder to your Google Drive root first.')
    print('Or use Option B to clone from GitHub.')

In [ ]:
# Install dependencies (PyTorch is pre-installed in Colab)
!pip install -q tiktoken safetensors tqdm datasets huggingface_hub fastapi uvicorn pydantic requests aiohttp

In [ ]:
# Verify GPU and dependencies
import torch
import sys
sys.path.insert(0, '/content/cerebro')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU.')

## Step 1: Configure Training

Choose your model tier and training parameters below.

In [ ]:
# ===== TRAINING CONFIGURATION =====

MODEL_TIER = "nano"       # nano (1.5B), core (7B), pro (13B), ultra (34B), sovereign (70B)
EPOCHS = 1                 # Number of training epochs
MAX_STEPS = 500            # Max steps (None = use preset default)
LEARNING_RATE = None       # Override LR (None = use preset default)
BATCH_SIZE = None          # Override batch size (None = use preset default)

# Dataset to download and train on (from catalog or HF Hub ID)
DATASET = "openhermes"     # Options: alpaca, ultrachat, openhermes, fineweb, wikipedia, ...
MAX_SAMPLES = 10000        # Max samples to download (None = all)

print(f'Model: {MODEL_TIER}')
print(f'Epochs: {EPOCHS}')
print(f'Dataset: {DATASET}')
print(f'Max samples: {MAX_SAMPLES}')

## Step 2: Download Dataset

Downloads the selected dataset from HuggingFace Hub.

In [ ]:
import os
os.chdir('/content/cerebro')

# Download the dataset
from pipeline.downloader import download_dataset

print(f'Downloading {DATASET}...')
result = download_dataset(
    dataset=DATASET,
    output_dir='datasets/raw',
    max_samples=MAX_SAMPLES,
)

if result:
    print(f'Downloaded: {result}')
else:
    print('Download failed or returned no data.')

## Step 3: Train Cerebro

Runs the full training pipeline: detect data → tokenize → train → save checkpoint.

In [ ]:
import sys
sys.path.insert(0, '/content/cerebro')

from pipeline.runner import PipelineConfig, run_pipeline

config = PipelineConfig(
    model_preset=MODEL_TIER,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    max_steps=MAX_STEPS,
    device='cuda',
    datasets_dir='datasets',
    output_dir='checkpoints',
)

print(config.summary())
print('\nStarting training...\n')

result = run_pipeline(config)
print(result.summary())

## Step 4: Save Checkpoint to Google Drive

Copies the trained model checkpoint to your Google Drive for persistence.

In [ ]:
import shutil
import os

CHECKPOINT_DIR = '/content/cerebro/checkpoints'
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/cerebro_checkpoints'

if os.path.exists(CHECKPOINT_DIR):
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    for fname in os.listdir(CHECKPOINT_DIR):
        src = os.path.join(CHECKPOINT_DIR, fname)
        dst = os.path.join(DRIVE_CHECKPOINT_DIR, fname)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(src) / 1e6
            print(f'Saved: {fname} ({size_mb:.1f} MB)')
    print(f'\nCheckpoints saved to Google Drive: {DRIVE_CHECKPOINT_DIR}')
else:
    print('No checkpoints found.')

## Step 5: Generate Text (Optional)

Test the trained model with a simple text generation prompt.

In [ ]:
# Optional: Generate text from the trained model
# Uncomment to run:

# from cerebro.config import CerebroConfig
# from cerebro.model.cerebro_model import CerebroModel
# from cerebro.tokenizer.tokenizer import CerebroTokenizer
# import torch
# 
# config = CerebroConfig.from_name(MODEL_TIER)
# model = CerebroModel(config)
# model.load_state_dict(torch.load('/content/cerebro/checkpoints/final/model.safetensors'))
# model.cuda()
# model.eval()
# 
# tokenizer = CerebroTokenizer()
# prompt = "The future of artificial intelligence is"
# input_ids = tokenizer.encode(prompt)
# output = model.generate(input_ids, max_new_tokens=50)
# print(tokenizer.decode(output))

## Model Tier Reference

| Tier | Params | Min GPU VRAM | Training Time (est.) |
|------|--------|-------------|---------------------|
| **nano** | 1.5B | 8 GB (T4) | ~1-2 hrs for 500 steps |
| **core** | 7B | 16 GB (T4 with GC) | ~4-6 hrs for 500 steps |
| **pro** | 13B | 24 GB (V100/A100) | ~8-12 hrs for 500 steps |
| **ultra** | 34B | 40 GB (A100) | Needs A100 subscription |
| **sovereign** | 70B | 80 GB (A100) | Needs Colab Pro+ |

**For Colab Free (T4 GPU, ~15GB VRAM):** Use `nano` or `core` with gradient checkpointing.

## Troubleshooting

- **Out of memory:** Reduce `BATCH_SIZE` or switch to `nano` tier
- **Slow training:** Check GPU is active: `!nvidia-smi`
- **Dataset not found:** Try a different dataset or check HuggingFace Hub
- **Colab disconnects:** Save checkpoints frequently; Colab Free has a ~4-12 hour limit